In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\KIIT\Desktop\Parkinson_EEG")
RES = BASE / "results1" / "SanDiego"
OUT = RES / "features_extracted"
OUT.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT)

# ============ Feature Functions ============
def hjorth_params(sig):
    diff1 = np.diff(sig)
    diff2 = np.diff(diff1)

    var0 = np.var(sig)
    var1 = np.var(diff1)
    var2 = np.var(diff2)

    activity = var0
    mobility = np.sqrt(var1 / var0) if var0 > 0 else 0
    complexity = (
        np.sqrt(var2 / var1) / mobility
        if var1 > 0 and mobility > 0 else 0
    )
    return activity, mobility, complexity


def extract_features(segment):
    # ensure segment is 1-D
    segment = np.asarray(segment).flatten()

    mean = np.mean(segment)
    std = np.std(segment)
    minv = np.min(segment)
    maxv = np.max(segment)

    act, mob, comp = hjorth_params(segment)

    return [
        mean, std, minv, maxv,
        act, mob, comp
    ]


# ============ Processing Wrapper ============
def process_npz(npz_path, tag):
    print(f"\nProcessing {tag} ->", npz_path.name)

    data = np.load(npz_path, allow_pickle=True)
    X_raw = data["X"]
    y = data["y"]
    subjects = data["subjects"]

    print("Input segments:", X_raw.shape)

    X_features = []

    for seg in X_raw:
        feats = extract_features(seg)
        X_features.append(feats)

    X_features = np.array(X_features, dtype=float)

    print("Final feature matrix:", X_features.shape)

    save_path = OUT / f"{tag}_features.npz"
    np.savez(save_path,
             X=X_features,
             y=y,
             subjects=subjects,
             feature_desc="mean,std,min,max,Hjorth(activity,mobility,complexity)"
            )

    print("Saved ->", save_path)
    return X_features


# ============ Run for both pipelines ============
caseA = RES / "features_caseA_ICA.npz"
caseB = RES / "features_caseB_Bandpass.npz"

if caseA.exists():
    process_npz(caseA, "caseA_ICA")

if caseB.exists():
    process_npz(caseB, "caseB_Bandpass")

print("\n✓ Feature extraction completed successfully.")

Output folder: C:\Users\KIIT\Desktop\Parkinson_EEG\results1\SanDiego\features_extracted

Processing caseA_ICA -> features_caseA_ICA.npz
Input segments: (4521, 205)
Final feature matrix: (4521, 7)
Saved -> C:\Users\KIIT\Desktop\Parkinson_EEG\results1\SanDiego\features_extracted\caseA_ICA_features.npz

Processing caseB_Bandpass -> features_caseB_Bandpass.npz
Input segments: (9064, 205)
Final feature matrix: (9064, 7)
Saved -> C:\Users\KIIT\Desktop\Parkinson_EEG\results1\SanDiego\features_extracted\caseB_Bandpass_features.npz

✓ Feature extraction completed successfully.
